# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbas72O5/flyrank-ml-internship_week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The Rule: I am prioritizing pages that are "Champions" (Average Position < 20) and are currently in a "Down" trend. The score is calculated by multiplying the page's Impressions by a binary flag for High Visibility. This ensures we focus on high-traffic assets where a decline is most expensive.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# CODE CELL for Section 1
import pandas as pd
import numpy as np

# Load the starter data
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Define our logic components
is_down = df['trend_direction'].str.lower().eq('down')
is_champion = df['avg_position'] < 20
is_stale = df['days_since_last_update'] > 180

print(f"Candidates for Champion Decay: {len(df[is_down & is_champion])}")

## 2. Build the ranked queue (writes the CSV)


*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Calculate the Score: Only score pages that are actually 'Down'
# Priority = Impressions * (1 if Position < 20 else 0)
df['baseline_score'] = is_down.astype(int) * is_champion.astype(int) * df['impressions_90d']

# Assign Reason Codes and Action Labels
df['reason_code'] = 'CHAMPION_DECAY_RISK'
df.loc[is_stale & is_champion, 'reason_code'] = 'STALE_CHAMPION'
df['action_label'] = 'REFRESH_CONTENT'

# Create the ranked queue (Top 100)
baseline_queue = df.sort_values('baseline_score', ascending=False).head(100)

# Ensure the output directory exists
os.makedirs("../outputs", exist_ok=True)

# Write to CSV
baseline_queue[['content_id', 'baseline_score', 'reason_code', 'action_label']].to_csv("../outputs/baseline_action_score.csv", index=False)

print("Ranked queue written to work/outputs/baseline_action_score.csv")
# Displaying top 5 for verification
display(baseline_queue[['content_id', 'baseline_score', 'reason_code', 'avg_position']].head(5))

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ID [16751]: Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: The keyword volume dropped globally (e.g., a holiday search term in January).
ID [16514]: Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: A new competitor tool launched that makes our text content obsolete regardless of a refresh.
ID [7021]: Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: This was a news-related spike that is naturally returning to a "normal" baseline.
ID [21268]: Action: Refresh. Code: STALE_CHAMPION. Wrong if: The page is a "evergreen" reference that doesn't need updates despite its age.
ID [11489]: Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: Technical SEO issue (like a broken CSS) is causing the drop, not the content quality.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.